In [1]:
import json
from datasets import load_dataset

In [ ]:
dataset = load_dataset("CAiRE/ASCEND")
with open("outputs/code_switch_timestamps.json", "r") as f:
    code_switch_timestamps = json.load(f)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00003.parquet:   0%|          | 0.00/317M [00:00<?, ?B/s]

main/train-00001-of-00003.parquet:   0%|          | 0.00/367M [00:00<?, ?B/s]

main/train-00002-of-00003.parquet:   0%|          | 0.00/328M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

main/validation-00000-of-00001.parquet:   0%|          | 0.00/107M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9869 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1315 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1130 [00:00<?, ? examples/s]

In [ ]:
metadata = {}

for split in code_switch_timestamps:

    metadata[split] = []

    for item in code_switch_timestamps[split]:

        idx = item["index"]
        sample = dataset[split][idx]

        duration = item["duration"]
        switches = item["switches"]

        switch_metadata = []

        for switch in switches:

            switch_time = switch["switch"]

            switch_metadata.append({
                "timestamp": switch_time,
                "direction": f'{switch["from"]}->{switch["to"]}',
                "relative_position": round(switch_time / duration, 3)
            })

        inter_switch_intervals = []

        for i in range(1, len(switches)):
            interval = (switches[i]["switch"] - switches[i - 1]["switch"])
            inter_switch_intervals.append(round(interval, 3))

        metadata[split].append({
            "index": idx,
            "speaker_id": sample["original_speaker_id"],
            "session_id": sample["session_id"],
            "duration": duration,
            "num_switches": len(switches),
            "switches": switch_metadata,
            "inter_switch_intervals": inter_switch_intervals
        })

with open("outputs/code_switch_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)